In [8]:
import numpy as np
import time

# CG podle zadání
def cg_assignment_version(A, b, x0=None, eps=1e-8, max_it=1000):
    n = len(b)
    x = np.zeros_like(b) if x0 is None else x0.copy()
    r = b - A @ x
    d = r.copy()
    delta_new = r.T @ r
    norm_r0A = np.sqrt(r.T @ A @ r)
    matvecs, vecvecs, k = 1, 2, 0
    while k < max_it:
        norm_rA = np.sqrt(r.T @ A @ r)
        matvecs += 1; vecvecs += 1
        if norm_rA / norm_r0A <= eps:
            break
        Ad = A @ d; matvecs += 1
        alpha = delta_new / (d.T @ Ad); vecvecs += 1
        x = x + alpha * d
        r = r - alpha * Ad
        delta_old = delta_new
        delta_new = r.T @ r; vecvecs += 1
        beta = delta_new / delta_old; vecvecs += 1
        d = r + beta * d
        k += 1
    return x, k, matvecs, vecvecs

In [9]:
# CG z přednášky
def cg_lecture_version(A, b, x0=None, eps=1e-8, max_it=1000):
    n = len(b)
    x = np.zeros_like(b) if x0 is None else x0.copy()
    r = b - A @ x
    d = r.copy()
    delta_new = r.T @ r
    delta_0 = delta_new
    matvecs, vecvecs, k = 1, 1, 0
    while k < max_it and delta_new > (eps**2) * delta_0:
        Ad = A @ d; matvecs += 1
        alpha = delta_new / (d.T @ Ad); vecvecs += 1
        x = x + alpha * d
        r = r - alpha * Ad
        delta_old = delta_new
        delta_new = r.T @ r; vecvecs += 1
        beta = delta_new / delta_old; vecvecs += 1
        d = r + beta * d
        k += 1
    return x, k, matvecs, vecvecs

In [10]:
# Testování a porovnání
np.random.seed(42)
n = 300
A = np.random.randn(n, n)
A = A @ A.T + n * np.eye(n)
b = np.random.randn(n)
x0 = np.zeros_like(b)
x_exact = np.linalg.solve(A, b)

start1 = time.perf_counter()
x1, it1, m1, v1 = cg_assignment_version(A, b, x0)
end1 = time.perf_counter()

start2 = time.perf_counter()
x2, it2, m2, v2 = cg_lecture_version(A, b, x0)
end2 = time.perf_counter()

print("CG podle zadání:")
print(f" Iterací: {it1}")
print(f" Čas: {end1 - start1:.6f} s")
print(f" Mat-vec: {m1}, Vec-vec: {v1}")
print(f" ||x1 - x*|| = {np.linalg.norm(x1 - x_exact):.2e}")

print("\nCG z přednášky:")
print(f" Iterací: {it2}")
print(f" Čas: {end2 - start2:.6f} s")
print(f" Mat-vec: {m2}, Vec-vec: {v2}")
print(f" ||x2 - x*|| = {np.linalg.norm(x2 - x_exact):.2e}")

print("\nRozdíl mezi x1 a x2:", np.linalg.norm(x1 - x2))

CG podle zadání:
 Iterací: 20
 Čas: 0.001700 s
 Mat-vec: 42, Vec-vec: 83
 ||x1 - x*|| = 1.12e-10

CG z přednášky:
 Iterací: 19
 Čas: 0.008553 s
 Mat-vec: 20, Vec-vec: 58
 ||x2 - x*|| = 2.88e-10

Rozdíl mezi x1 a x2: 2.2663405154771474e-10


In [1]:
import numpy as np      # Importuje knihovnu NumPy pro práci s maticemi a vektory
import time             # Importuje knihovnu time pro měření času

# ──────────────────────────────
# CG podle zadání – úplná verze
# ──────────────────────────────
def cg_assignment_version(A, b, x0=None, eps=1e-8, max_it=1000):
    n = len(b)                                     # Délka vektoru b
    x = np.zeros_like(b) if x0 is None else x0.copy()  # Počáteční aproximace x0 (ve výchozím případě nulový vektor)
    r = b - A @ x                                  # Reziduum r₀ = b - Ax₀
    d = r.copy()                                   # Počáteční směr d₀ = r₀
    delta_new = r.T @ r                            # delta_new = rᵀr
    norm_r0A = np.sqrt(r.T @ A @ r)                # ||r₀||_A = √(rᵀAr)
    matvecs, vecvecs, k = 1, 2, 0                  # Počáteční počty operací: 1x mat-vec, 2x vec-vec, 0 iterací

    while k < max_it:                              # Hlavní iterace
        norm_rA = np.sqrt(r.T @ A @ r)             # ||r||_A v každé iteraci
        matvecs += 1
        vecvecs += 1

        if norm_rA / norm_r0A <= eps:              # Kriterium konvergence
            break

        Ad = A @ d                                 # Mat-vec násobení: A*d
        matvecs += 1

        alpha = delta_new / (d.T @ Ad)             # Výpočet koeficientu alpha
        vecvecs += 1

        x = x + alpha * d                          # Nová aproximace xₖ₊₁
        r = r - alpha * Ad                         # Nové reziduum rₖ₊₁

        delta_old = delta_new                      # Uložení starého skaláru
        delta_new = r.T @ r                        # Nový skalár rᵀr
        vecvecs += 1

        beta = delta_new / delta_old               # Výpočet koeficientu beta
        vecvecs += 1

        d = r + beta * d                           # Aktualizace směru dₖ₊₁
        k += 1                                      # Inkrementace počtu iterací

    return x, k, matvecs, vecvecs                  # Vrací řešení x, počet iterací, počet mat-vec a vec-vec

# ────────────────────────────────────
# CG z přednášky – jednodušší verze
# ────────────────────────────────────
def cg_lecture_version(A, b, x0=None, eps=1e-8, max_it=1000):
    n = len(b)
    x = np.zeros_like(b) if x0 is None else x0.copy()
    r = b - A @ x
    d = r.copy()
    delta_new = r.T @ r
    delta_0 = delta_new                            # Uložíme si počáteční hodnotu delta₀ pro srovnání v kritériu
    matvecs, vecvecs, k = 1, 1, 0

    while k < max_it and delta_new > (eps**2) * delta_0:
        Ad = A @ d
        matvecs += 1

        alpha = delta_new / (d.T @ Ad)
        vecvecs += 1

        x = x + alpha * d
        r = r - alpha * Ad

        delta_old = delta_new
        delta_new = r.T @ r
        vecvecs += 1

        beta = delta_new / delta_old
        vecvecs += 1

        d = r + beta * d
        k += 1

    return x, k, matvecs, vecvecs

# ────────────────────────────────────────
# Generování testovacích dat a porovnání
# ────────────────────────────────────────
np.random.seed(42)                                 # Pro opakovatelnost výsledků
n = 300                                            # Rozměr matice A
A = np.random.randn(n, n)                          # Náhodná matice A
A = A @ A.T + n * np.eye(n)                        # Úprava A → zajištění symetrické pozitivně definitní matice
b = np.random.randn(n)                             # Pravá strana b
x0 = np.zeros_like(b)                              # Počáteční aproximace x₀ = nulový vektor
x_exact = np.linalg.solve(A, b)                    # Přesné řešení pomocí LU

# Spuštění verze podle zadání
start1 = time.perf_counter()
x1, it1, m1, v1 = cg_assignment_version(A, b, x0)
end1 = time.perf_counter()

# Spuštění zjednodušené verze
start2 = time.perf_counter()
x2, it2, m2, v2 = cg_lecture_version(A, b, x0)
end2 = time.perf_countser()

# Výpiseme výsledků – verze podle zadání
print("CG podle zadání:")
print(f" Iterací: {it1}")
print(f" Čas: {end1 - start1:.6f} s")
print(f" Mat-vec: {m1}, Vec-vec: {v1}")
print(f" ||x1 - x*|| = {np.linalg.norm(x1 - x_exact):.2e}")

# Výpis výsledků – verze z přednášky
print("\nCG z přednášky:")
print(f" Iterací: {it2}")
print(f" Čas: {end2 - start2:.6f} s")
print(f" Mat-vec: {m2}, Vec-vec: {v2}")
print(f" ||x2 - x*|| = {np.linalg.norm(x2 - x_exact):.2e}")

# Rozdíl mezi výsledky
print("\nRozdíl mezi x1 a x2:", np.linalg.norm(x1 - x2))


CG podle zadání:
 Iterací: 20
 Čas: 0.000982 s
 Mat-vec: 42, Vec-vec: 83
 ||x1 - x*|| = 1.12e-10

CG z přednášky:
 Iterací: 19
 Čas: 0.002530 s
 Mat-vec: 20, Vec-vec: 58
 ||x2 - x*|| = 2.88e-10

Rozdíl mezi x1 a x2: 2.2663405154771474e-10
